In [ ]:
from sheet_1 import simulate
from qiskit import QuantumCircuit
import numpy as np




# Gates

Qiskit can transpile all gates to a series of basis gates, in our case the one qubit rotation gate 'u' and the two qubit gate 'cx'. These two gates were defined in 'apply_u_on_state.py' and 'apply_cx_on_state.py'. 

## u gate

In [ ]:
import numpy as np

def apply_u_on_state(state: np.ndarray, u: np.ndarray, acting_on: int) -> np.ndarray:
    """Applies a single-qubit gate to the given state vector. The gate is defined 
    by the 2x2 matrix u, and it acts on the qubit specified by acting_on"""
    number_of_qubits=state.ndim
    acting_on=number_of_qubits-acting_on-1
    old_indices = [i for i in range(number_of_qubits)]
    new_indices = old_indices.copy()
    new_indices[acting_on] = 51
    result=np.einsum(u,[51,acting_on],state,old_indices,new_indices)
    return result

## cx gate

In [ ]:
import numpy as np

def apply_cx_on_state(state: np.ndarray, cx: np.ndarray, acting_on1: int, acting_on2: int) -> np.ndarray:
    '''Applies a CNOT gate to the given state vector. The CNOT gate is defined 
    by the 4x4 matrix cx, and it acts on the qubits specified by acting_on1 (control) 
    and acting_on2 (target). The state vector is modified in-place.'''
    number_of_qubits=state.ndim
    acting_on1=number_of_qubits-1-acting_on1
    acting_on2=number_of_qubits-1-acting_on2
    cx_matrix=np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
    cx = np.reshape(cx_matrix,(2,2,2,2))
    old_indices = [i for i in range(number_of_qubits)]
    new_indices = old_indices.copy()
    new_indices[acting_on1] = 51
    new_indices[acting_on2] = 50
    result=np.einsum(cx,[51,50,acting_on1,acting_on2],state,old_indices,new_indices)
    return result

# Simulator

The simulator uses the two gates 'u' and 'cx'. It simulates the, to basisgates 'u' and 'cx' transpiled, quantum circuit and returns the final state vector.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate, UGate, CXGate
from qiskit_aer import AerSimulator
from sheet_1.apply_cx_on_state import apply_cx_on_state
from sheet_1.apply_u_on_state import apply_u_on_state
import numpy as np
from qiskit.quantum_info import Statevector



def simulate(qc: QuantumCircuit, parameters: dict | None = None) -> np.ndarray:
    """Simulates the given quantum circuit and returns the final state vector."""
    number_of_qubits=qc.num_qubits
    psi=np.zeros(2**number_of_qubits)
    psi[0]=1
    psi_converted = np.reshape(psi,[2]*number_of_qubits)

    for instr, qargs, cargs in qc.data:
        qubit_indices = [qc.find_bit(q).index for q in qargs]

        if instr.name == 'cx':
            cx=np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
            psi_converted=apply_cx_on_state(psi_converted,cx,qubit_indices[0], qubit_indices[1])
        elif instr.name == 'u':
            u=UGate(instr.params[0],instr.params[1],instr.params[2]).to_matrix()
            psi_converted=apply_u_on_state(psi_converted,u,qubit_indices[0])
        else:
            continue
    
    psi_final = np.reshape(np.asarray(psi_converted), (2**number_of_qubits,))
    return psi_final












The simulator was tested with a simple and a random circuit. 